# Fine-tune DINOv2 cho Nhận diện Địa danh (Kaggle End-to-End)
Notebook này được thiết kế theo dạng **"Run All"**. Nó chứa toàn bộ vòng đời huấn luyện mô hình: từ thiết lập môi trường, tải Data, định nghĩa mô hình (Hugging Face DINOv2), vòng lặp train và cuối cùng là Inference in ra xác suất Top-K.

> **💡 Lưu ý trên Kaggle:** Bạn cần bật tuỳ chọn truy cập Internet (Internet: On) và GPU (Accelerator: T4 x2 hoặc P100) ở góc phải màn hình của Kaggle Notebook trước khi chạy.


## 1. Cài đặt các thư viện cần thiết


In [ ]:
!pip install -q transformers Pillow tqdm scikit-learn


## 2. Import thư viện


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')


## 3. Cấu hình Tham số và Môi trường
Dưới đây là các cấu hình đường dẫn. Khi bạn Add Data của bạn vào Kaggle, chúng thường nằm ở `/kaggle/input/tên-dataset/...`. Hãy sửa `TRAIN_DIR` và `VAL_DIR` cho đúng nhé!


In [ ]:
# === THIẾT LẬP KAGGLE ===
# Trên Kaggle, thư mục working là nơi có quyền ghi/đọc, còn input là nơi chứa data (chỉ đọc)

TRAIN_DIR = "/kaggle/working/dataset/train"
VAL_DIR = "/kaggle/working/dataset/val"

BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 5e-5
MODEL_NAME = "facebook/dinov2-base"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# --- TẠO DUMMY DATA CHO RUN ALL ---
# !! BẠN HÃY XÓA ĐOẠN NÀY VÀ TRỎ TRAIN_DIR, VAL_DIR VỀ DATA CỦA BẠN KHI DÙNG THẬT !!
print("Tạo dữ liệu dummy để kiểm tra Run All...")
for split in [TRAIN_DIR, VAL_DIR]:
    os.makedirs(f"{split}/Hanoi", exist_ok=True)
    os.makedirs(f"{split}/HoChiMinh", exist_ok=True)
    Image.new('RGB', (224, 224), color='red').save(f"{split}/Hanoi/img1.jpg")
    Image.new('RGB', (224, 224), color='blue').save(f"{split}/HoChiMinh/img2.jpg")


## 4. Dataset và DataLoader


In [ ]:
class DINOv2DatasetProcessor:
    def __init__(self, model_name):
        self.processor = AutoImageProcessor.from_pretrained(model_name)

    def transform(self, image):
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)
        if image.mode != "RGB":
            image = image.convert("RGB")
            
        inputs = self.processor(images=image, return_tensors="pt")
        return inputs['pixel_values'].squeeze(0)

def create_dataloaders(train_dir, val_dir, batch_size, model_name, num_workers=2):
    processor = DINOv2DatasetProcessor(model_name=model_name)
    
    train_dataset = datasets.ImageFolder(root=train_dir, transform=processor.transform)
    val_dataset = datasets.ImageFolder(root=val_dir, transform=processor.transform)
    
    # Kaggle CPU num_workers thường bằng 2 là tối ưu
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    
    class_to_idx = train_dataset.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    
    return train_loader, val_loader, idx_to_class, len(class_to_idx)


## 5. Khởi tạo Model (Hugging Face)


In [ ]:
def create_model(num_classes, model_name, id2label=None, label2id=None):
    model = AutoModelForImageClassification.from_pretrained(
        model_name,
        num_labels=num_classes,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True
    )
    return model


## 6. Vòng lặp Huấn luyện (Training Loop)


In [ ]:
def train_model():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🖥️ Thiết bị huấn luyện: {device}")

    print("Đang khởi tạo DataLoader...")
    train_loader, val_loader, idx_to_class, num_classes = create_dataloaders(
        TRAIN_DIR, VAL_DIR, BATCH_SIZE, MODEL_NAME
    )
    class_to_idx = {v: k for k, v in idx_to_class.items()}

    print(f"Tìm thấy {num_classes} classes: {idx_to_class}")
    
    print("Đang khởi tạo Model...")
    model = create_model(num_classes, MODEL_NAME, id2label=idx_to_class, label2id=class_to_idx)
    model.to(device)

    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0

    # Để test code chạy được trên Kaggle ngay lập tức mà không mất thời gian, 
    # dummy data sẽ chỉ chạy 1 epoch cho mục đích minh họa. 
    # Khi dùng data thật, hãy dùng biến EPOCHS.
    RUN_EPOCHS = 1 if num_classes == 2 and "Hanoi" in idx_to_class.values() else EPOCHS

    for epoch in range(RUN_EPOCHS):
        # --- TRAIN ---
        model.train()
        train_loss = 0.0
        train_correct = 0
        total_train = 0
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{RUN_EPOCHS} [Train]")
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            outputs = model(images)
            logits = outputs.logits
            loss = criterion(logits, labels)
            
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(logits, 1)
            total_train += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            loop.set_postfix(loss=loss.item())

        train_loss = train_loss / total_train
        train_acc = train_correct / total_train

        # --- VALIDATION ---
        model.eval()
        val_loss = 0.0
        val_correct = 0
        total_val = 0
        
        with torch.no_grad():
            loop_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{RUN_EPOCHS} [Val]")
            for images, labels in loop_val:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                logits = outputs.logits
                loss = criterion(logits, labels)
                
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(logits, 1)
                total_val += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
                loop_val.set_postfix(loss=loss.item())

        val_loss = val_loss / total_val
        val_acc = val_correct / total_val

        print(f"👉 Epoch {epoch+1}: Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        # --- LƯU CHECKPOINT TỐT NHẤT ---
        if val_acc >= best_val_acc:
            best_val_acc = val_acc
            ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pth")
            print(f"⭐ Checkpoint mới tốt nhất! Lưu tại: {ckpt_path}")
            checkpoint = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "best_val_acc": best_val_acc,
                "idx_to_class": idx_to_class,
                "num_classes": num_classes,
                "model_name": MODEL_NAME
            }
            torch.save(checkpoint, ckpt_path)
            
    print("🎉 HUẤN LUYỆN HOÀN TẤT!")


## 7. Bắt đầu Huấn luyện!


In [ ]:
train_model()


## 8. Inference (Dự đoán Top-K)


In [ ]:
def predict_image(image_path, checkpoint_path, top_k=5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 1. Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    idx_to_class = checkpoint['idx_to_class']
    num_classes = checkpoint['num_classes']
    model_name = checkpoint['model_name']
    class_to_idx = {v: k for k, v in idx_to_class.items()}
    
    # 2. Khởi tạo Model & Processor
    model = create_model(num_classes, model_name, id2label=idx_to_class, label2id=class_to_idx)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    processor = AutoImageProcessor.from_pretrained(model_name)
    
    # 3. Đọc và tiền xử lý ảnh
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    pixel_values = inputs['pixel_values'].to(device)
    
    # 4. Nội suy (Inference)
    with torch.no_grad():
        logits = model(pixel_values).logits
        probabilities = F.softmax(logits, dim=-1)
        
    # 5. Top-K
    top_prob, top_indices = torch.topk(probabilities, min(top_k, num_classes))
    
    top_prob = top_prob.squeeze().tolist()
    top_indices = top_indices.squeeze().tolist()
    if not isinstance(top_prob, list):
        top_prob = [top_prob]
        top_indices = [top_indices]
        
    print(f"📸 Kết quả dự đoán cho ảnh: {image_path}")
    for prob, idx in zip(top_prob, top_indices):
        print(f"  🎯 [{idx_to_class[idx]}]: {prob*100:.2f}%")


## 9. Thử nghiệm dự đoán trên 1 ảnh


In [ ]:
# Chạy thử dự đoán trên một bức ảnh dummy vừa tạo
test_image_path = f"{VAL_DIR}/Hanoi/img1.jpg"
best_ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pth")

if os.path.exists(test_image_path) and os.path.exists(best_ckpt_path):
    predict_image(test_image_path, best_ckpt_path, top_k=2)
else:
    print("Vui lòng huấn luyện model hoặc cung cấp đường dẫn ảnh đúng.")
